# 2. Hand-made feature vectors for words and expressions

Before learning vectors with neural networks, we build them **by hand**, with simple algorithms.
Each word or expression (an **n-gram**) found in the reports of Notebook 1 receives a vector of
four positions:

| Position | Meaning | How it is computed |
|---|---|---|
| 1 | the n-gram is a **disease** | lookup in the disease dictionary |
| 2 | the n-gram is a **symptom** | lookup in the symptom dictionary |
| 3 | the n-gram is a **negation word** | lookup in the negation cue list |
| 4 | the n-gram is **in negation scope** | a negation cue appears earlier in the same sentence, with no terminator ("but", "however") in between |

*Metaphor:* the lexicon is a **field guide**. For each creature (expression) it tells you what kind of
creature it is. Position 4 is not in the guide: it depends on **where** the creature was spotted.

The n-grams are needed to **find** meanings ("yellow tongue" is a symptom, "tongue" alone is not).
At the end, the properties found for n-grams are reflected back to the **words** inside them, so
that each word gets its own vector.

Run this notebook after `01_generate_reports.ipynb`.

In [1]:
import re
from collections import defaultdict

import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SENTENCES_FILE = "sentences.csv"
ANNOTATIONS_FILE = "annotations.csv"
LEXICON_FILE = "lexicon.csv"
VECTORS_FILE = "vectors.csv"
CONTEXT_WORDS_FILE = "context_words.csv"
WORD_VECTORS_FILE = "word_vectors.csv"

# Longest n-gram considered ("trembling on the little finger" has 5 tokens).
MAX_N = 5

FEATURES = ["f1_disease", "f2_symptom", "f3_negation", "f4_neg_scope"]

## Load the sentences and build the dictionaries

Each dictionary maps a **surface form** (as written in the text) to its **canonical** name.

In [2]:
sentences = pd.read_csv(SENTENCES_FILE)
annotations = pd.read_csv(ANNOTATIONS_FILE)
lexicon = pd.read_csv(LEXICON_FILE)

def dictionary(kind):
    rows = lexicon[lexicon["type"] == kind]
    return dict(zip(rows["surface"], rows["canonical"]))

DISEASE_DICT = dictionary("disease")
SYMPTOM_DICT = dictionary("symptom")
NEGATION_CUES = set(dictionary("negation"))
TERMINATORS = set(dictionary("terminator"))

print(f"{len(sentences)} sentences")
print(f"{len(DISEASE_DICT)} disease forms, {len(SYMPTOM_DICT)} symptom forms")
print(f"negation cues: {sorted(NEGATION_CUES)}")
print(f"terminators:   {sorted(TERMINATORS)}")

184 sentences
12 disease forms, 18 symptom forms
negation cues: ['absence', 'absent', 'denies', 'free of', 'negative', 'no', 'not', 'ruled out', 'without']
terminators:   ['but', 'however']


## Tokenize and extract n-grams

Tokens are lowercase alphanumeric sequences; punctuation is discarded. An n-gram is a sequence of
`n` consecutive tokens, identified by its start position.

In [3]:
def tokenize(sentence):
    return re.findall(r"[a-z0-9]+", sentence.lower())

def extract_ngrams(tokens, n):
    return [(i, " ".join(tokens[i:i + n])) for i in range(len(tokens) - n + 1)]

example = sentences["sentence"].iloc[0]
tokens = tokenize(example)
print(example)
print(tokens)
for n in (1, 2, 3):
    print(n, extract_ngrams(tokens, n))

The patient presents with paralysis.
['the', 'patient', 'presents', 'with', 'paralysis']
1 [(0, 'the'), (1, 'patient'), (2, 'presents'), (3, 'with'), (4, 'paralysis')]
2 [(0, 'the patient'), (1, 'patient presents'), (2, 'presents with'), (3, 'with paralysis')]
3 [(0, 'the patient presents'), (1, 'patient presents with'), (2, 'presents with paralysis')]


## Negation scope

A negation cue can have more than one token ("free of", "ruled out"). For every token position we
mark whether it is **inside the scope** of a cue: it comes after the end of a cue and before the end of
the sentence or a terminator. This is a tiny version of the NegEx algorithm.

In [4]:
def cue_spans(tokens):
    """(start, end) of each negation cue; end is exclusive."""
    spans = []
    for n in range(1, MAX_N + 1):
        for start, gram in extract_ngrams(tokens, n):
            if gram in NEGATION_CUES:
                spans.append((start, start + n))
    return spans

def negation_scope(tokens):
    """in_scope[i] is True if token i is governed by a preceding negation cue."""
    in_scope = [False] * len(tokens)
    for _, end in cue_spans(tokens):
        for i in range(end, len(tokens)):
            if tokens[i] in TERMINATORS:
                break
            in_scope[i] = True
    return in_scope

for s in ["No signs of paralysis.", "The patient is free of chest pain.",
          "Examination reveals yellowish tongue.", "Signs of paralysis were not observed."]:
    t = tokenize(s)
    print(f"{s:40} {[w for w, inside in zip(t, negation_scope(t)) if inside]}")

No signs of paralysis.                   ['signs', 'of', 'paralysis']
The patient is free of chest pain.       ['chest', 'pain']
Examination reveals yellowish tongue.    []
Signs of paralysis were not observed.    ['observed']


The last example is the **hard mode** case: the cue comes *after* the term, so the term is not in
scope. Keep it in mind for the evaluation below.

## The feature vector of each n-gram occurrence

In [5]:
def vectorize(sentence):
    tokens = tokenize(sentence)
    in_scope = negation_scope(tokens)
    rows = []
    for n in range(1, MAX_N + 1):
        for start, gram in extract_ngrams(tokens, n):
            rows.append({
                "start": start,
                "n": n,
                "ngram": gram,
                "f1_disease": int(gram in DISEASE_DICT),
                "f2_symptom": int(gram in SYMPTOM_DICT),
                "f3_negation": int(gram in NEGATION_CUES),
                "f4_neg_scope": int(in_scope[start]),
            })
    return rows

pd.DataFrame(vectorize("The patient is free of chest pain.")).query("n <= 2")

,start,n,ngram,f1_disease,f2_symptom,f3_negation,f4_neg_scope
0,0,1,the,0,0,0,0
1,1,1,patient,0,0,0,0
2,2,1,is,0,0,0,0
3,3,1,free,0,0,0,0
4,4,1,of,0,0,0,0
5,5,1,chest,0,0,0,1
6,6,1,pain,0,0,0,1
7,0,2,the patient,0,0,0,0
8,1,2,patient is,0,0,0,0
9,2,2,is free,0,0,0,0


In [6]:
vector_rows = []
for _, s in sentences.iterrows():
    for row in vectorize(s["sentence"]):
        vector_rows.append({"report_id": s["report_id"], "sentence_id": s["sentence_id"],
                            "section": s["section"], **row})

vectors = pd.DataFrame(vector_rows)
vectors.to_csv(VECTORS_FILE, index=False)

print(f"{len(vectors)} n-gram occurrences saved to {VECTORS_FILE}")
vectors.groupby(FEATURES).size().rename("occurrences").reset_index()

3997 n-gram occurrences saved to vectors.csv


,f1_disease,f2_symptom,f3_negation,f4_neg_scope,occurrences
0,0,0,0,0,3245
1,0,0,0,1,579
2,0,0,1,0,68
3,0,1,0,0,31
4,0,1,0,1,30
5,1,0,0,0,25
6,1,0,0,1,19


Most occurrences have the vector `[0, 0, 0, 0]` or `[0, 0, 0, 1]`: they are words or pieces of
expressions that the field guide does not know. Only a few n-grams "light up" positions 1–3.

## Longest match

When two dictionary hits overlap, we keep the **longest** one. In the current lexicon no entry is
nested inside another, so this step changes nothing yet. It becomes essential as soon as we add a
shorter entry such as "infection", which would otherwise also be found inside "bacterial infection".

In [7]:
def longest_matches(hits):
    """Keep dictionary hits that are not contained in a longer hit of the same sentence."""
    kept = []
    for sentence_id, group in hits.groupby("sentence_id"):
        spans = sorted(group.itertuples(), key=lambda r: -r.n)
        taken = set()
        for r in spans:
            positions = set(range(r.start, r.start + r.n))
            if not positions & taken:
                taken |= positions
                kept.append(r.Index)
    return hits.loc[sorted(kept)]

hits = vectors[(vectors["f1_disease"] == 1) | (vectors["f2_symptom"] == 1)]
terms = longest_matches(hits)
print(f"{len(hits)} dictionary hits, {len(terms)} after longest match")

105 dictionary hits, 105 after longest match


## From vectors back to facts

Two rules are enough to read the reports:

* `f2 = 1` and `f4 = 0` → the symptom is **present** (`f1` for diseases);
* `f2 = 1` and `f4 = 1` → the symptom is **absent**.

We compare the extracted facts with the ground truth saved by Notebook 1.

In [8]:
extracted = terms.assign(
    canonical=[DISEASE_DICT.get(g) or SYMPTOM_DICT.get(g) for g in terms["ngram"]],
    type=["disease" if d else "symptom" for d in terms["f1_disease"]],
    predicted=["absent" if f4 else "present" for f4 in terms["f4_neg_scope"]],
)[["sentence_id", "ngram", "canonical", "type", "predicted"]]

truth = annotations[annotations["type"].isin(["symptom", "disease"])]
evaluation = truth.merge(extracted, on=["sentence_id", "canonical", "type"], how="outer")
evaluation["predicted"] = evaluation["predicted"].fillna("missed")
evaluation["polarity"] = evaluation["polarity"].fillna("spurious")

accuracy = (evaluation["polarity"] == evaluation["predicted"]).mean()
print(f"Accuracy: {accuracy:.1%} over {len(evaluation)} facts")
pd.crosstab(evaluation["polarity"], evaluation["predicted"], rownames=["truth"], colnames=["predicted"])

Accuracy: 100.0% over 105 facts


predicted,absent,present
truth,,
absent,49,0
present,0,56


In [9]:
# The sentences where the simple rules fail (empty unless hard mode is on).
errors = evaluation[evaluation["polarity"] != evaluation["predicted"]]
errors.merge(sentences, on="sentence_id")[["sentence", "canonical", "polarity", "predicted"]]

,sentence,canonical,polarity,predicted


**Try it:** set `POST_NEGATION = True` in Notebook 1, run it again, and then run this notebook.
Sentences such as "Signs of paralysis were not observed." break the rule, because position 4 only
looks **backward**.

## From occurrences to expression vectors: a "passport"

So far each **occurrence** has a vector. Averaging the vectors of all occurrences of the same
expression gives one vector per expression, like a passport. Positions 1–3 stay 0 or 1, because they
come from the field guide. Position 4 becomes a **negation rate**: how often the expression was seen
inside a negation.

This table (one row per expression, one column per property) is our first **embedding-like**
representation.

In [10]:
known = vectors[(vectors[["f1_disease", "f2_symptom", "f3_negation"]].sum(axis=1) > 0)]

passports = (
    known.groupby("ngram")[FEATURES].mean()
    .join(known.groupby("ngram").size().rename("count"))
    .sort_values(["f1_disease", "f2_symptom", "f3_negation", "count"], ascending=False)
    .round(2)
)
passports

,f1_disease,f2_symptom,f3_negation,f4_neg_scope,count
ngram,,,,,
infection of bacterial origin,1.0,0.0,0.0,0.25,8
bacterial infectious process,1.0,0.0,0.0,0.33,6
bacterial infection,1.0,0.0,0.0,0.60,5
bite related deficit,1.0,0.0,0.0,0.40,5
bite deficit,1.0,0.0,0.0,0.25,4
deficit caused by bite,1.0,0.0,0.0,0.67,3
fight injury,1.0,0.0,0.0,0.33,3
altercation injury,1.0,0.0,0.0,1.00,2
fight related trauma,1.0,0.0,0.0,0.50,2


In [11]:
# The same idea per canonical term: all synonyms share one passport.
by_canonical = known.assign(
    canonical=[DISEASE_DICT.get(g) or SYMPTOM_DICT.get(g) or g for g in known["ngram"]]
).groupby("canonical")

canonical_passports = (
    by_canonical[FEATURES].mean()
    .join(by_canonical.size().rename("count"))
    .sort_values(["f1_disease", "f2_symptom", "count"], ascending=False)
    .round(2)
)
canonical_passports

,f1_disease,f2_symptom,f3_negation,f4_neg_scope,count
canonical,,,,,
bacterial infection,1.0,0.0,0.0,0.37,19
bite deficit,1.0,0.0,0.0,0.42,12
fights,1.0,0.0,0.0,0.57,7
viral infection,1.0,0.0,0.0,0.50,6
yellow tongue,0.0,1.0,0.0,0.41,17
trembling on the little finger,0.0,1.0,0.0,0.54,13
chest pain,0.0,1.0,0.0,0.44,9
member loss,0.0,1.0,0.0,0.25,8
paralysis,0.0,1.0,0.0,0.75,8


## From n-grams back to words

The passports describe **expressions**. To get one vector per **word**, we reflect what was found
for the n-grams back onto the words inside them:

| Position | A word gets 1 if... | Example |
|---|---|---|
| 1 | it is inside a **disease** n-gram found in the sentence | "viral" in "infection of viral origin" |
| 2 | it is inside a **symptom** n-gram found in the sentence | "tongue" in "yellow tinted tongue" |
| 3 | it is inside a **negation cue** | both "free" and "of" in "free of" |
| 4 | it is **in negation scope** (already a word property) | "tongue" in "No signs of yellow tongue." |

*Metaphor:* a **team jersey**. The dictionary spots a team (the n-gram), and every player (word) of
that team wears its jersey. The same player can wear another team's jersey in another sentence,
or no jersey at all.

Symptom and disease hits come from `terms` (after longest match), so a word inherits only from the
term that actually won.

In [12]:
def reflect(hits):
    """Set of (sentence_id, token position) covered by the n-gram hits."""
    return {(r.sentence_id, i) for r in hits.itertuples() for i in range(r.start, r.start + r.n)}

disease_positions = reflect(terms[terms["f1_disease"] == 1])
symptom_positions = reflect(terms[terms["f2_symptom"] == 1])
negation_positions = reflect(vectors[vectors["f3_negation"] == 1])

# Words are the 1-grams; position 4 is already a property of each token.
word_vectors = vectors[vectors["n"] == 1].drop(columns="n").rename(columns={"ngram": "word"})
position = list(zip(word_vectors["sentence_id"], word_vectors["start"]))
word_vectors["f1_disease"] = [int(p in disease_positions) for p in position]
word_vectors["f2_symptom"] = [int(p in symptom_positions) for p in position]
word_vectors["f3_negation"] = [int(p in negation_positions) for p in position]
word_vectors.to_csv(WORD_VECTORS_FILE, index=False)
print(f"{len(word_vectors)} word occurrences saved to {WORD_VECTORS_FILE}")

# One sentence, word by word: n-gram vectors (1-grams only) versus reflected word vectors.
demo = sentences.loc[sentences["sentence"].str.contains("free of|denies"), "sentence_id"].iloc[0]
print(sentences.loc[sentences["sentence_id"] == demo, "sentence"].iloc[0])
pd.concat(
    {"as 1-gram": vectors[(vectors["sentence_id"] == demo) & (vectors["n"] == 1)].set_index("ngram")[FEATURES],
     "as word": word_vectors[word_vectors["sentence_id"] == demo].set_index("word")[FEATURES]},
    axis=1,
)

1166 word occurrences saved to word_vectors.csv
The patient denies trembling on the little finger.


as 1-gram                                        as word  \
          f1_disease f2_symptom f3_negation f4_neg_scope f1_disease   
the                0          0           0            0          0   
patient            0          0           0            0          0   
denies             0          0           1            0          0   
trembling          0          0           0            1          0   
on                 0          0           0            1          0   
the                0          0           0            1          0   
little             0          0           0            1          0   
finger             0          0           0            1          0   

                                               
          f2_symptom f3_negation f4_neg_scope  
the                0           0            0  
patient            0           0            0  
denies             0           1            0  
trembling          1           0            1  
on                 1           0            1  
the                1           0            1  
little             1           0            1  
finger             1           0            1

As a 1-gram, "tongue" is unknown to the field guide; as a word, it inherits the symptom column of
the n-gram that contains it.

### Word passports

Averaging the occurrences of each word gives one vector per word. As before, the positions become
**rates**: how often the word was part of a disease, of a symptom, of a negation cue, and inside a
negation scope. Darker cells are higher rates.

In [13]:
word_passports = (
    word_vectors.groupby("word")[FEATURES].mean()
    .join(word_vectors.groupby("word").size().rename("count"))
)

SHOW = ["tongue", "yellow", "yellowish", "green", "strange", "chest", "pain", "finger", "anger",
        "infection", "viral", "bite", "fight", "no", "of", "patient"]
(word_passports.loc[[w for w in SHOW if w in word_passports.index]]
    .style.background_gradient(cmap="Blues", vmin=0, vmax=1, subset=FEATURES)
    .format("{:.2f}", subset=FEATURES))

,f1_disease,f2_symptom,f3_negation,f4_neg_scope,count
word,,,,,
tongue,0.00,0.50,0.00,0.21,34
yellow,0.00,0.92,0.00,0.38,13
yellowish,0.00,1.00,0.00,0.40,5
green,0.00,0.00,0.00,0.00,3
strange,0.00,0.00,0.00,0.00,2
chest,0.00,0.55,0.00,0.27,11
pain,0.00,0.75,0.00,0.38,8
finger,0.00,0.81,0.00,0.44,16
anger,0.00,0.45,0.00,0.27,11


Reading the rows:

* "yellowish" was always part of a symptom (1.00) and "yellow" almost always (0.92): once, it was a
  raincoat.
* "tongue" was part of a symptom only **half** of the time (0.50). The other half it was a healthy
  tongue or a language, and the vector cannot tell which: both look like `[0, 0, 0, 0]`.
* "infection", "viral", "bite" and "fight" lean on the disease column; "fight" loses more weight to
  its everyday uses.
* "no" is a pure negation word. "of" wears a bit of every jersey: it sits inside diseases
  ("infection of viral origin"), symptoms ("shaking of the pinky finger"), the cue "free of", and is
  in negation scope half of the time. Function words get blurry vectors; content words get sharp ones.
* "green" and "strange" are all zeros: the field guide knows nothing about them.

### One word, many occurrences

The rates of a word are an average over sentences. Listing them shows where each number comes from.


In [14]:
# Where the rates of one word come from: its distinct sentences and their word vectors.
WORD = "tongue"   # try: "yellow", "chest", "of", "no"

occurrences = word_vectors[word_vectors["word"] == WORD].merge(sentences[["sentence_id", "sentence"]], on="sentence_id")
occurrences.groupby(["sentence", *FEATURES]).size().rename("times").reset_index().sort_values(FEATURES, ascending=False)

,sentence,f1_disease,f2_symptom,f3_negation,f4_neg_scope,times
7,No signs of yellow tinted tongue.,0,1,0,1,1
8,No signs of yellow tongue.,0,1,0,1,1
11,The patient denies yellow tinted tongue.,0,1,0,1,2
12,The patient denies yellowish tongue.,0,1,0,1,1
13,The patient is free of yellow tinted tongue.,0,1,0,1,1
15,The patient presents without yellowish tongue.,0,1,0,1,1
1,Examination reveals yellow tongue.,0,1,0,0,1
2,Examination reveals yellowish tongue.,0,1,0,0,1
3,Findings include yellow tinted tongue.,0,1,0,0,1
4,Findings include yellow tongue.,0,1,0,0,1


## Same word, different company

The word vector of "tongue" tells that it was part of a symptom in some sentences and nothing in
the others. Notebook 1 used it in four different contexts:

| Context | Example | What the reader understands |
|---|---|---|
| symptom, present | "Findings include yellow tongue." | a sick zombie |
| symptom, absent | "No signs of yellow tongue." | the symptom was checked and is absent |
| normal finding | "Tongue color: green." | a healthy zombie |
| everyday | "The patient speaks in a strange tongue." | a language, not a body part |

The first two contexts are recognized from the word vector: columns 2 and 4. For the other two, the
word vector is `[0, 0, 0, 0]`: the algorithm only knows that the word is **outside any term**. Below,
those occurrences get their label from the ground truth saved by Notebook 1 (`context_words.csv`),
so we can see what the word vector is missing.

In [15]:
context_words = pd.read_csv(CONTEXT_WORDS_FILE)
BORROWED = sorted(set(context_words["word"]))
truth_context = dict(zip(zip(context_words["sentence_id"], context_words["word"]), context_words["context"]))

def context_of(r):
    """The context read from the word vector; outside any term, the ground truth."""
    if r.f1_disease or r.f2_symptom:
        return ("disease" if r.f1_disease else "symptom") + (", absent" if r.f4_neg_scope else ", present")
    return truth_context.get((r.sentence_id, r.word), "outside any term")

words = word_vectors[word_vectors["word"].isin(BORROWED)].copy()
words["context"] = [context_of(r) for r in words.itertuples()]
pd.crosstab(words["word"], words["context"])

context,"disease, absent","disease, present",everyday,normal finding,"symptom, absent","symptom, present"
word,,,,,,
anger,0,0,1,5,3,2
bite,5,7,0,2,0,0
chest,0,0,3,2,3,3
control,0,0,1,2,2,1
fight,2,3,4,0,0,0
finger,0,0,1,2,7,6
infection,6,11,1,0,0,0
limb,0,0,0,1,2,5
limbs,0,0,0,3,2,0


Every row of this table is one word with a single passport, but its occurrences are spread over
several columns. The word passport averages all of them: it mixes a sick zombie, a healthy zombie
and a zombie that speaks a strange language.

### The company it keeps

The context sentences cannot be told apart by the field guide, because none of them contains a
known expression. They differ in their **neighbors**. Below, each occurrence of one borrowed word
collects the words in a small window around it, and the neighbors are counted per context.

In [16]:
WORD = "tongue"   # try: "chest", "finger", "anger", "member", "fight"
WINDOW = 2

sentence_tokens = {s: tokenize(t) for s, t in zip(sentences["sentence_id"], sentences["sentence"])}

def neighbors(sentence_id, i):
    t = sentence_tokens[sentence_id]
    return t[max(0, i - WINDOW):i] + t[i + 1:i + 1 + WINDOW]

company = (
    words[words["word"] == WORD]
    .assign(neighbor=lambda d: [neighbors(s, i) for s, i in zip(d["sentence_id"], d["start"])])
    .explode("neighbor")
)
pd.crosstab(company["neighbor"], company["context"])

context,everyday,normal finding,"symptom, absent","symptom, present"
neighbor,,,,
a,6,0,0,0
are,0,0,0,1
at,3,0,0,0
color,0,1,0,0
covers,0,1,0,0
denies,0,0,1,0
dry,0,1,0,0
evident,0,0,0,1
green,0,3,0,0


Read the table by columns: each column is a first **context vector** for the word, one position per
neighbor. For "tongue", the symptom columns are dominated by "yellow", "yellowish" and "tinted"; the
normal findings bring "green", "healthy" and "dry"; the everyday uses bring "sharp", "strange",
"mother" and "zombish". The absent column, however, mostly repeats the present one: the cue shows up
only when it is next to the term ("denies", "without"); in "No signs of yellow tongue." the "no" is
outside the window. Neighbors reveal the **sense** of a word; position 4 (negation scope) is still
needed for the **polarity**.

Two lessons for embeddings:

* a word does not have **one** meaning; the neighbors reveal which sense is being used;
* the more text, the clearer the columns. Raise `VARIANTS_PER_PATIENT` in Notebook 1 to see the
  counts grow while the pattern stays the same.

## Where to go next

* **More positions** (a triage form with more checkboxes): pathogen dictionary, hedge cues
  ("possible", "suspected"), history cues, section of the report, distance to the nearest cue.
* **Weighting:** TF-IDF, as in `information-retrieval/basics/cabot/ir-03-tfidf.ipynb`.
* **Company it keeps:** extend the neighbor counts above to every word, giving a word × neighbor
  co-occurrence matrix, and compare rows with cosine similarity.
* **Jersey + company:** concatenate each word passport with its neighbor counts; the first part
  says what the word was part of, the second part says in which sense it was used.
* **A map:** PPMI + SVD in 2 dimensions, the bridge to dense embeddings.
* **Learned embeddings:** a tiny word2vec on the generated sentences, compared with these passports.